# Part 4: Simple Scheduler

So far, we can generate tokens for a single request, control sampling, and keep multiple requests alive with their own tokens and KV caches.

But just having multiple requests is not enough.

If several requests are waiting for the GPU, we need something that decides:

* which requests should run now
* which requests should wait
* when a new request can start
* when a finished request should be removed

That component is the **scheduler**.

At a high level:

```text
incoming requests
       ↓
    waiting
       ↓
   scheduler
       ↓
    running
       ↓
 model forward
       ↓
update request state
       ↓
 scheduler again
```

The scheduler sits between our request state and the model execution loop.

For now, we will keep it intentionally simple. We are **not building continuous batching yet**.

We will first build a basic scheduler and understand exactly what problem it solves. Once that works, we can extend the same design into continuous batching later.


In [1]:
#basic initilization
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# We are using the same small Qwen model from the previous parts.
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load the tokenizer.
# The tokenizer converts text -> token IDs and token IDs -> text.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the actual language model onto the GPU.
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,device_map="cuda")

# We are only doing inference, so the model should stay in evaluation mode.
model.eval()

device = "cuda"

print("Model loaded:", model_name)
print("Device:", device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda


In [2]:
#scheduler types of queues

waiting = []
running = []
finished = []


In [3]:
# 2 simple requests
request_a = {
    "id": "A",
    "prompt": "The capital of France is",
    "generated_tokens": [],
    "finished": False
}

request_b = {
    "id": "B",
    "prompt": "The largest planet is",
    "generated_tokens": [],
    "finished": False
}

In [4]:
#routing request to the waiting queue
waiting.append(request_a)
waiting.append(request_b)

print("Waiting:", [req["id"] for req in waiting])
print("Running:", [req["id"] for req in running])

Waiting: ['A', 'B']
Running: []


## Scheduler State

The scheduler needs to know which requests are waiting, running, or finished.

```python
waiting = []
running = []
finished = []
```

We can add incoming requests to the waiting queue:

```python
request_a = {
    "id": "A",
    "prompt": "The capital of France is",
    "generated_tokens": [],
    "finished": False
}

request_b = {
    "id": "B",
    "prompt": "The largest planet is",
    "generated_tokens": [],
    "finished": False
}

waiting.append(request_a)
waiting.append(request_b)

print("Waiting:", [req["id"] for req in waiting])
print("Running:", [req["id"] for req in running])
```

Output:

```text
Waiting: ['A', 'B']
Running: []
```

At this point, both requests exist, but neither has been scheduled to run yet.


In [6]:
#FIFO scheduling: (first in first out) shifting requests from waiting to running queue

max_running=1 #this tells how much requests a gpu can run at a time

if len(running)<max_running and len(waiting)>0:
  request= waiting.pop(0)
  running.append(request)

print("Waiting:", [req["id"] for req in waiting])
print("Running:", [req["id"] for req in running])



Waiting: ['B']
Running: ['A']


## Moving a request from `waiting` to `running`

Right now both requests are sitting in the waiting queue:

```text
waiting = [A, B]
running = []
```

The scheduler's first job is to decide which request can start running.

For now, we will keep the rule very simple:

> Only one request can be in `running` at a time.

```python
max_running = 1

if len(running) < max_running and len(waiting) > 0:
    request = waiting.pop(0)
    running.append(request)

print("Waiting:", [req["id"] for req in waiting])
print("Running:", [req["id"] for req in running])
```

Output:

```text
Waiting: ['B']
Running: ['A']
```

### What happened?

This condition:

```python
len(running) < max_running
```

checks whether there is currently space to admit another request.

Since:

```text
len(running) = 0
max_running = 1
```

there is one free slot.

We also check:

```python
len(waiting) > 0
```

because there is nothing to schedule if the waiting queue is empty.

Then:

```python
request = waiting.pop(0)
```

takes the request at index `0` from the waiting queue **and removes it from the list**.

Before:

```text
waiting = [A, B]
```

After:

```text
request = A
waiting = [B]
```

We then move that request into the running list:

```python
running.append(request)
```

giving us:

```text
waiting = [B]
running = [A]
```

So request `A` has gone through its first scheduler state transition:

```text
A: waiting -> running
```

Because we always take index `0`, requests are admitted in the same order they arrived.

This is **FIFO scheduling — First In, First Out**.

At this point, the scheduler is only deciding **who is allowed to run**. We still have not executed the model or generated any tokens yet.


In [7]:
#shifting request from running to finished

request = running.pop(0)
request["finished"] = True
finished.append(request)

print("Waiting:", [req["id"] for req in waiting])
print("Running:", [req["id"] for req in running])
print("Finished:", [req["id"] for req in finished])

Waiting: ['B']
Running: []
Finished: ['A']


In [8]:
#carrying out the fifo operation for 2nd request
if len(running) < max_running and len(waiting) > 0:
    request = waiting.pop(0)
    running.append(request)

print("Waiting:", [req["id"] for req in waiting])
print("Running:", [req["id"] for req in running])
print("Finished:", [req["id"] for req in finished])

Waiting: []
Running: ['B']
Finished: ['A']


## Finishing a request and freeing capacity

At this point:

```text
waiting = [B]
running = [A]
finished = []
```

Since `max_running = 1`, request `B` cannot start while `A` is still using the only running slot.

Suppose `A` finishes generation.

We move it out of `running`:

```python
request = running.pop(0)
request["finished"] = True
finished.append(request)

print("Waiting:", [req["id"] for req in waiting])
print("Running:", [req["id"] for req in running])
print("Finished:", [req["id"] for req in finished])
```

Output:

```text
Waiting: ['B']
Running: []
Finished: ['A']
```

Now `running` is empty, so the scheduler has capacity again.

We can run the same admission logic:

```python
if len(running) < max_running and len(waiting) > 0:
    request = waiting.pop(0)
    running.append(request)

print("Waiting:", [req["id"] for req in waiting])
print("Running:", [req["id"] for req in running])
print("Finished:", [req["id"] for req in finished])
```

Output:

```text
Waiting: []
Running: ['B']
Finished: ['A']
```

So the request states changed like this:

```text
A: waiting -> running -> finished
B: waiting -----------> running
```

The key idea is that **finishing a request frees scheduler capacity**.

```text
A finishes
    ↓
running slot becomes free
    ↓
scheduler checks waiting queue
    ↓
B is admitted
```

Right now we are performing these steps manually.

The next step is to move this admission logic into a real `schedule()` function so the scheduler can make these decisions for us.


In [45]:
#reseting everything

from transformers import DynamicCache

waiting = []
running = []
finished = []

max_running = 2

In [46]:
#request state

def create_request(request_id, prompt, max_new_tokens=10):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    return {
        "id": request_id,
        "input_ids": input_ids,
        "generated_tokens": [],
        "cache": DynamicCache(),
        "prefilled": False,
        "max_new_tokens": max_new_tokens,
        "finished": False
    }

In [47]:
#creating few requests

waiting.append(
    create_request("A", "The capital of France is")
)

waiting.append(
    create_request("B", "The largest planet in our solar system is")
)

waiting.append(
    create_request("C", "The fastest land animal is")
)

In [48]:
#defining a simple FIFO scheduling function

def schedule():
    while len(running) < max_running and len(waiting) > 0:
        running.append(waiting.pop(0))

In [49]:
# Run one inference step for one request

@torch.no_grad()
def run_request(request):

    # First time this request runs:
    # prefill the prompt and predict the first generated token
    if not request["prefilled"]:
        outputs = model(
            input_ids=request["input_ids"],
            past_key_values=request["cache"],
            use_cache=True
        )

        request["prefilled"] = True

    # From the second generated token onward:
    # feed only the previous generated token and reuse the KV cache
    else:
        last_token = torch.tensor(
            [[request["generated_tokens"][-1]]],
            device=device
        )

        outputs = model(
            input_ids=last_token,
            past_key_values=request["cache"],
            use_cache=True
        )

    # Save the newly updated KV cache
    request["cache"] = outputs.past_key_values

    next_token_logits = outputs.logits[:, -1, :]
    next_token = torch.argmax(next_token_logits, dim=-1).item()

    request["generated_tokens"].append(next_token)

    # Stop if EOS or token limit reached
    if (
        next_token == tokenizer.eos_token_id
        or len(request["generated_tokens"]) >= request["max_new_tokens"]
    ):
        request["finished"] = True

In [50]:
def run_scheduler():

    while len(waiting) > 0 or len(running) > 0:

        # Fill free running slots using FIFO
        schedule()

        # Give every running request one inference step
        for request in running:
            run_request(request)

        # Remove finished requests
        for request in running[:]:
            if request["finished"]:
                running.remove(request)
                finished.append(request)

In [51]:
run_scheduler()

In [52]:
for request in finished:
    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True
    )

    print(f"Request {request['id']}")
    print(f"Output: {text}")
    print(f"Cache length: {request['cache'].get_seq_length()}")
    print("-" * 40)

Request A
Output:  Paris. It is the largest city in Europe and
Cache length: 14
----------------------------------------
Request B
Output:  ____
A. Venus
B. Mars

Cache length: 17
----------------------------------------
Request C
Output:  the cheetah, which can run at a
Cache length: 14
----------------------------------------


## Building a Simple FIFO Scheduler

So far, we have already learned how to:

* keep multiple requests alive independently
* store generated tokens for each request
* maintain a separate KV cache for each request
* perform prefill once
* perform decode one token at a time afterward

Now we can put those pieces behind a scheduler.

The scheduler's job is not to generate tokens itself.

Its job is to decide:

```text
which requests are waiting
which requests are allowed to run
when a finished request should leave
when another waiting request can take its place
```

For this first version, we will use a very simple **FIFO — First In, First Out** policy.

---

## Resetting the scheduler state

We will start fresh.

```python
from transformers import DynamicCache

waiting = []
running = []
finished = []

max_running = 2
```

We now have three request states:

```text
waiting
    requests that have arrived but are not currently active

running
    requests currently admitted by the scheduler

finished
    requests whose generation has completed
```

For this example:

```python
max_running = 2
```

means the scheduler is allowed to keep at most two requests active at once.

Important: in this simple implementation, this does **not** mean the two requests are executed together in one GPU batch.

It only means:

```text
at most 2 requests can belong to the running set
```

We will deal with actual batched execution later.

---

## Creating request state

Every request needs to carry everything required to resume generation later.

```python
def create_request(request_id, prompt, max_new_tokens=10):
    input_ids = tokenizer(
        prompt,
        return_tensors="pt"
    ).input_ids.to(device)

    return {
        "id": request_id,
        "input_ids": input_ids,
        "generated_tokens": [],
        "cache": DynamicCache(),
        "prefilled": False,
        "max_new_tokens": max_new_tokens,
        "finished": False
    }
```

Each request stores:

```text
id
    lets us identify the request

input_ids
    tokenized prompt

generated_tokens
    tokens generated so far

cache
    that request's KV cache

prefilled
    whether the prompt has already gone through prefill

max_new_tokens
    maximum number of tokens this request may generate

finished
    whether generation for this request has ended
```

The important idea is that the scheduler does not store generation state globally.

Each request carries its own state.

---

## Creating a few requests

We can now simulate three incoming requests.

```python
waiting.append(
    create_request(
        "A",
        "The capital of France is"
    )
)

waiting.append(
    create_request(
        "B",
        "The largest planet in our solar system is"
    )
)

waiting.append(
    create_request(
        "C",
        "The fastest land animal is"
    )
)
```

Initially:

```text
waiting  = [A, B, C]
running  = []
finished = []
```

Since the requests were inserted in the order:

```text
A -> B -> C
```

our FIFO scheduler will preserve that order.

---

# FIFO admission

The first scheduler function only handles admission.

```python
def schedule():

    while len(running) < max_running and len(waiting) > 0:

        running.append(waiting.pop(0))
```

There are two conditions:

```python
len(running) < max_running
```

means:

> there is still space in the running set.

And:

```python
len(waiting) > 0
```

means:

> there is actually another request waiting.

Because we use:

```python
waiting.pop(0)
```

the request that has waited the longest is taken first.

For:

```text
waiting = [A, B, C]
max_running = 2
```

one call to:

```python
schedule()
```

produces:

```text
waiting = [C]

running = [A, B]
```

This is our FIFO scheduling policy.

---

# Running one inference step

The scheduler decides **who runs**.

We still need something that performs the actual model work for one request.

```python
@torch.no_grad()
def run_request(request):

    # First time this request runs:
    # prefill the prompt and predict the first generated token
    if not request["prefilled"]:

        outputs = model(
            input_ids=request["input_ids"],
            past_key_values=request["cache"],
            use_cache=True
        )

        request["prefilled"] = True

    # From the second generated token onward:
    # feed only the previous generated token and reuse the KV cache
    else:

        last_token = torch.tensor(
            [[request["generated_tokens"][-1]]],
            device=device
        )

        outputs = model(
            input_ids=last_token,
            past_key_values=request["cache"],
            use_cache=True
        )

    # Save the newly updated KV cache
    request["cache"] = outputs.past_key_values

    # We only care about the logits for the newest position
    next_token_logits = outputs.logits[:, -1, :]

    # For now, use greedy decoding
    next_token = torch.argmax(
        next_token_logits,
        dim=-1
    ).item()

    request["generated_tokens"].append(next_token)

    # Stop if EOS or token limit reached
    if (
        next_token == tokenizer.eos_token_id
        or len(request["generated_tokens"]) >= request["max_new_tokens"]
    ):
        request["finished"] = True
```

There are two execution paths inside this function.

### First scheduler step for a request

Suppose request A has never run before.

Then:

```python
request["prefilled"] == False
```

so the full prompt enters the model:

```text
"The capital of France is"
            ↓
          prefill
            ↓
       KV cache built
            ↓
     first token predicted
```

After this:

```python
request["prefilled"] = True
```

so the prompt will never be processed from scratch again.

---

## Later scheduler steps

On the next scheduler iteration, that request enters the `else` branch.

We take:

```python
request["generated_tokens"][-1]
```

which means:

> the most recently generated token

and turn it into:

```python
[[token_id]]
```

so the tensor shape is:

```text
[batch_size, sequence_length]

[1, 1]
```

Only that single token enters the model.

The previously computed prompt and token history already exists inside:

```python
request["cache"]
```

So execution becomes:

```text
previous KV cache
      +
latest token
      ↓
    decode
      ↓
updated KV cache
      ↓
next token
```

This is exactly the prefill/decode behavior we built earlier, but now every request owns its own version of it.

---

# Why `run_request()` generates only one token

One call to:

```python
run_request(request)
```

does **one inference step**.

It does not generate all ten tokens.

For example:

```text
call 1
    prefill
    predict token 1

call 2
    decode token 1
    predict token 2

call 3
    decode token 2
    predict token 3

...
```

The request itself decides when it has generated enough tokens:

```python
len(request["generated_tokens"]) >= request["max_new_tokens"]
```

The scheduler will repeatedly call `run_request()` until that happens.

This separation is important:

```text
run_request()
    performs one unit of inference work

scheduler
    decides when the request gets another unit of work
```

---

# The scheduler execution loop

Now we can connect everything together.

```python
def run_scheduler():

    while len(waiting) > 0 or len(running) > 0:

        # Fill free running slots using FIFO
        schedule()

        # Give every running request one inference step
        for request in running:
            run_request(request)

        # Remove finished requests
        for request in running[:]:

            if request["finished"]:

                running.remove(request)
                finished.append(request)
```

Then:

```python
run_scheduler()
```

This is the first complete scheduler control loop we have built.

---

# Understanding the outer `while`

The scheduler keeps running while:

```python
len(waiting) > 0 or len(running) > 0
```

In simple terms:

> keep the engine alive while there is still work somewhere.

If:

```text
waiting = []
running = []
```

there is nothing left to do, so the scheduler stops.

---

# Walking through the scheduler

Initially:

```text
waiting  = [A, B, C]
running  = []
finished = []
```

Then:

```python
schedule()
```

runs.

Because:

```python
max_running = 2
```

we get:

```text
waiting  = [C]
running  = [A, B]
finished = []
```

Now the execution loop does:

```python
run_request(A)
run_request(B)
```

On their first step:

```text
A -> prefill -> token 1
B -> prefill -> token 1
```

Neither request has generated 10 tokens yet.

So the next scheduler iteration begins.

`schedule()` sees:

```text
running = [A, B]
```

and because both slots are already occupied, it does nothing.

Then:

```text
A -> decode -> token 2
B -> decode -> token 2
```

The same cycle continues:

```text
scheduler iteration 1
A -> token 1
B -> token 1

scheduler iteration 2
A -> token 2
B -> token 2

scheduler iteration 3
A -> token 3
B -> token 3

...
```

Eventually A and B reach their stopping condition:

```python
len(generated_tokens) >= 10
```

so both become:

```python
request["finished"] = True
```

They are removed from:

```text
running
```

and moved into:

```text
finished
```

giving:

```text
waiting  = [C]
running  = []
finished = [A, B]
```

The scheduler starts its next iteration.

`schedule()` now sees two free slots and one waiting request.

So:

```text
C: waiting -> running
```

and C begins generation.

Eventually:

```text
waiting  = []
running  = []
finished = [A, B, C]
```

The outer `while` condition becomes false and the scheduler stops.

---

# Printing the results

After the scheduler finishes, we can inspect every completed request.

```python
for request in finished:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True
    )

    print(f"Request {request['id']}")
    print(f"Output: {text}")
    print(
        f"Generated tokens: "
        f"{len(request['generated_tokens'])}"
    )
    print(
        f"Cache length: "
        f"{request['cache'].get_seq_length()}"
    )

    print("-" * 40)
```

Example output:

```text
Request A
Output: Paris. It is the largest city in Europe and
Generated tokens: 10
Cache length: 14
----------------------------------------

Request B
Output: ____ A. Venus B. Mars
Generated tokens: 10
Cache length: 17
----------------------------------------

Request C
Output: the cheetah, which can run at a
Generated tokens: 10
Cache length: 14
----------------------------------------
```

---

# Understanding the KV-cache lengths

Request A generated 10 tokens but ended with:

```text
cache length = 14
```

Its prompt contained approximately 5 tokens.

We might initially expect:

```text
5 prompt tokens
+
10 generated tokens
=
15
```

but that is not what happens.

During prefill:

```text
5 prompt tokens enter model
↓
cache length = 5
↓
token 1 is predicted
```

Token 1 has only been **predicted**.

It has not entered the model yet.

During the next decode:

```text
token 1 enters model
↓
cache length = 6
↓
token 2 is predicted
```

This continues until:

```text
token 9 enters model
↓
cache length = 14
↓
token 10 is predicted
```

Generation then stops because:

```text
generated_tokens = 10
```

Token 10 is never fed back through another model forward pass.

Therefore:

```text
final cache length
=
prompt length + generated tokens - 1
```

For A:

```text
5 + 10 - 1 = 14
```

For B:

```text
8 + 10 - 1 = 17
```

For C:

```text
5 + 10 - 1 = 14
```

So the KV-cache behavior is exactly what we expect.

---

# What we have actually built

We now have a working scheduler with:

```text
request state
        ↓
waiting queue
        ↓
FIFO scheduler
        ↓
running set
        ↓
prefill / decode step
        ↓
update request state
        ↓
remove finished requests
        ↓
admit the next waiting request
        ↓
repeat
```

The control flow is:

```text
schedule
   ↓
execute
   ↓
update
   ↓
finish/remove
   ↓
schedule again
```

This is the important scheduler mental model.

Production inference engines make each stage significantly more sophisticated, but this basic control loop remains recognizable.

---

## Current limitation: `running` does not mean batched execution

We currently have:

```python
for request in running:
    run_request(request)
```

If:

```text
running = [A, B]
```

the actual GPU execution is:

```text
model(A)

then

model(B)
```

These are two separate forward passes.

So:

```python
max_running = 2
```

currently means:

> two requests may be active at the same time from the scheduler's perspective.

It does **not** yet mean:

> execute two requests together in one model batch.

Eventually we want something conceptually closer to:

```text
[A, B]
   ↓
one batched model execution
```

That brings us toward continuous batching.

---

## Note about the generated text

The scheduler behavior in this section is correct, but one generated output clearly does not make semantic sense:

```text
Request B
Output: ____ A. Venus B. Mars
```

The prompt was:

```text
"The largest planet in our solar system is"
```

and the expected factual continuation should involve **Jupiter**, not Venus or Mars.

This section used `Qwen/Qwen2.5-0.5B-Instruct` while feeding plain tokenized text directly into the model rather than using the model's intended instruction/chat formatting.

Because of that, the model can interpret the input more like arbitrary text completion and produce strange continuations such as the multiple-choice-style output above.

Requests A and C happened to produce reasonable continuations:

```text
A -> Paris...
C -> the cheetah...
```

but that does not make the prompting setup ideal.

For this part, the goal was specifically to understand the **scheduler control flow**, so the strange generation does not affect the scheduling behavior we were testing.

From the next part onward, we will explicitly check the model and prompt-format compatibility before building further on the engine, so model behavior and scheduler behavior are not accidentally mixed together.

---

# Where we are now

We now have a simple but functioning **FIFO inference scheduler**:

```text
incoming requests
       ↓
waiting queue
       ↓
FIFO admission
       ↓
running requests
       ↓
one inference step per request
       ↓
finished requests removed
       ↓
new waiting requests admitted
       ↓
repeat
```

The biggest thing still missing is obvious:

```text
multiple active requests
        ≠
batched GPU execution
```

Right now:

```text
A -> model()
B -> model()
```


In [63]:
#Resource-aware scheduling - resent state

from transformers import DynamicCache

waiting = []
running = []
finished = []

# Maximum number of active requests
max_running = 2

# Maximum amount of token work we want to schedule
# in one scheduler iteration
max_tokens_per_step = 16



In [64]:
def create_request(request_id, prompt, max_new_tokens=10):

    input_ids = tokenizer(
        prompt,
        return_tensors="pt"
    ).input_ids.to(device)

    return {
        "id": request_id,

        # Original prompt
        "input_ids": input_ids,

        # Tokens produced so far
        "generated_tokens": [],

        # Each request owns its own KV cache
        "cache": DynamicCache(),

        # False until the prompt has gone through prefill
        "prefilled": False,

        "max_new_tokens": max_new_tokens,

        "finished": False
    }

In [65]:
waiting.append(
    create_request(
        "A",
        "The capital of France is"
    )
)

waiting.append(
    create_request(
        "B",
        "The largest planet in our solar system is"
    )
)

waiting.append(
    create_request(
        "C",
        "The fastest land animal is"
    )
)

In [66]:
#This is where our scheduler becomes resource-aware

def tokens_needed(request):

    # Prefill processes the entire prompt
    if not request["prefilled"]:
        return request["input_ids"].shape[1]

    # Decode processes only one new token
    return 1

In [67]:
#One inference step for one request

@torch.no_grad()
def run_request(request):

    # First execution = prefill
    if not request["prefilled"]:

        outputs = model(
            input_ids=request["input_ids"],
            past_key_values=request["cache"],
            use_cache=True
        )

        request["prefilled"] = True

    # Every execution after that = decode
    else:

        last_token = torch.tensor(
            [[request["generated_tokens"][-1]]],
            device=device
        )

        outputs = model(
            input_ids=last_token,
            past_key_values=request["cache"],
            use_cache=True
        )

    # Save updated KV cache
    request["cache"] = outputs.past_key_values

    # Logits for the newest position
    next_token_logits = outputs.logits[:, -1, :]

    # Greedy decoding for now
    next_token = torch.argmax(
        next_token_logits,
        dim=-1
    ).item()

    request["generated_tokens"].append(next_token)

    # Request-level stopping condition
    if (
        next_token == tokenizer.eos_token_id
        or
        len(request["generated_tokens"])
        >= request["max_new_tokens"]
    ):
        request["finished"] = True

In [68]:
#Resource-aware scheduler step

def schedule():

    scheduled_this_step = []

    token_budget = max_tokens_per_step


    # -----------------------------------
    # 1. Consider already-running requests
    # -----------------------------------

    for request in running:

        cost = tokens_needed(request)

        if cost <= token_budget:

            scheduled_this_step.append(request)

            token_budget -= cost


    # -----------------------------------
    # 2. Admit new requests if we have room
    # -----------------------------------

    index = 0

    while (
        index < len(waiting)
        and len(running) < max_running
    ):

        request = waiting[index]

        cost = tokens_needed(request)

        if cost <= token_budget:

            waiting.pop(index)

            running.append(request)

            scheduled_this_step.append(request)

            token_budget -= cost

        else:

            # Don't let one expensive request
            # block every request behind it
            index += 1


    # -----------------------------------
    # 3. Guarantee forward progress
    # -----------------------------------

    # Without chunked prefill, a prompt larger
    # than max_tokens_per_step would never fit.
    #
    # Until we implement chunked prefill later,
    # allow one oversized request if otherwise
    # no work could happen.

    if (
        len(scheduled_this_step) == 0
        and len(waiting) > 0
        and len(running) < max_running
    ):

        request = waiting.pop(0)

        running.append(request)

        scheduled_this_step.append(request)


    return scheduled_this_step

In [69]:
def run_scheduler():

    while len(waiting) > 0 or len(running) > 0:

        # Decide who gets model work this iteration
        scheduled_this_step = schedule()

        # Execute only the selected requests
        for request in scheduled_this_step:

            run_request(request)

        # Remove completed requests
        for request in running[:]:

            if request["finished"]:

                running.remove(request)

                finished.append(request)

In [70]:
run_scheduler()

In [71]:
for request in finished:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True
    )

    print(f"Request {request['id']}")

    print(f"Output: {text}")

    print(
        f"Generated tokens: "
        f"{len(request['generated_tokens'])}"
    )

    print(
        f"Cache length: "
        f"{request['cache'].get_seq_length()}"
    )

    print("-" * 50)

Request A
Output:  Paris. It is the largest city in Europe and
Generated tokens: 10
Cache length: 14
--------------------------------------------------
Request B
Output:  ____
A. Venus
B. Mars

Generated tokens: 10
Cache length: 17
--------------------------------------------------
Request C
Output:  the cheetah, which can run at a
Generated tokens: 10
Cache length: 14
--------------------------------------------------


# Resource-Aware Scheduling

Our first scheduler was intentionally simple.

It had three request states:

```text
waiting
running
finished
```

and a basic admission rule:

```python
while len(running) < max_running and len(waiting) > 0:
    running.append(waiting.pop(0))
```

This gave us a working FIFO scheduler:

```text
requests arrive
      ↓
waiting queue
      ↓
oldest request admitted first
      ↓
running
      ↓
one inference step
      ↓
finished requests leave
      ↓
repeat
```

That control flow is useful, but there is an important problem.

The scheduler only thinks in terms of **number of requests**.

For example:

```text
Request A → 5-token prompt
Request B → 5,000-token prompt
```

Our old scheduler sees:

```text
A = 1 request
B = 1 request
```

even though the amount of model work required for those two requests is completely different.

So now we add another idea:

> A scheduler should also consider how much work a request needs before deciding what gets execution time.

---

# Scheduling policy vs resource awareness

These are two different concepts.

## Scheduling policy

The scheduling policy answers:

> Which request should I prefer?

Examples include:

```text
FIFO / FCFS
priority scheduling
shortest-job-first style policies
deadline-aware scheduling
fair-share scheduling
```

Our original scheduler used FIFO:

```text
A arrives
B arrives
C arrives

order = A → B → C
```

---

# What does priority-aware mean?

A **priority-aware scheduler** assigns or receives an explicit priority for requests.

For example:

```text
A → priority 5
B → priority 1
C → priority 3
```

If smaller numbers mean higher priority:

```text
B runs before C
C runs before A
```

even if A arrived first.

Conceptually:

```python
request = {
    "id": "B",
    "priority": 1,
    ...
}
```

and the scheduler might order requests using that value.

Priority-aware scheduling can be useful for things such as:

```text
paid user vs free user
interactive request vs background request
urgent production traffic vs offline job
different internal services
```

But **we are not implementing priority scheduling here**.

Our scheduler still prefers arrival order.

---

# What are we actually implementing?

This version is:

```text
FIFO-biased
+
resource-aware
+
work-conserving
```

### FIFO-biased

We inspect older requests first.

### Resource-aware

We consider how much token work a request needs this iteration.

### Work-conserving

If an older request cannot use the remaining budget, we are willing to let another request that fits use it.

This means our scheduler is **not strict FIFO** anymore.

That is intentional.

---

# Why not strict FIFO?

Suppose:

```text
token budget = 10

A needs 5
B needs 8
C needs 1
```

Strict FIFO could do:

```text
A runs
remaining budget = 5

B needs 8
B does not fit

STOP
```

We would leave:

```text
5 tokens worth of capacity unused
```

even though C only needs one token.

That is unnecessary underutilization.

Instead, our scheduler can do:

```text
A → fits
B → does not fit right now
C → fits
```

so:

```text
scheduled = [A, C]
```

B has not been deleted or rejected.

It simply waits for another scheduler iteration.

---

# 1. Reset scheduler state

We start fresh:

```python
from transformers import DynamicCache

waiting = []
running = []
finished = []

# Maximum number of active requests
max_running = 2

# Maximum amount of token work that can be
# selected in one scheduler iteration
max_tokens_per_step = 16
```

These two limits mean different things.

```text
max_running = 2
```

means:

> At most two requests can currently belong to our active set.

While:

```text
max_tokens_per_step = 16
```

means:

> During one scheduler iteration, select at most roughly 16 tokens worth of work.

For now this is just a simple **token-work budget**.

It is not literally GPU memory measured in bytes.

---

# 2. Request state

Every request carries the state required to stop and resume generation.

```python
def create_request(request_id, prompt, max_new_tokens=10):

    input_ids = tokenizer(
        prompt,
        return_tensors="pt"
    ).input_ids.to(device)

    return {
        "id": request_id,

        # Original tokenized prompt
        "input_ids": input_ids,

        # Tokens generated so far
        "generated_tokens": [],

        # This request's KV cache
        "cache": DynamicCache(),

        # False until its prompt has gone
        # through the model once
        "prefilled": False,

        # Maximum generation length
        "max_new_tokens": max_new_tokens,

        # Becomes True when generation ends
        "finished": False
    }
```

Then we create three requests:

```python
waiting.append(
    create_request(
        "A",
        "The capital of France is"
    )
)

waiting.append(
    create_request(
        "B",
        "The largest planet in our solar system is"
    )
)

waiting.append(
    create_request(
        "C",
        "The fastest land animal is"
    )
)
```

Initially:

```text
WAITING
┌─────┬─────┬─────┐
│  A  │  B  │  C  │
└─────┴─────┴─────┘

RUNNING
[ empty ]

FINISHED
[ empty ]
```

From our actual run, their prompt lengths were effectively:

```text
A → 5 tokens
B → 8 tokens
C → 5 tokens
```

---

# 3. How much work does a request need?

The scheduler needs some way of estimating the cost of the **next inference step**.

For our current engine:

```python
def tokens_needed(request):

    # Prefill processes the entire prompt
    if not request["prefilled"]:
        return request["input_ids"].shape[1]

    # Decode processes only one new token
    return 1
```

Why?

---

## Prefill cost

For a new request:

```text
The capital of France is
```

all prompt tokens enter the model:

```text
[t1, t2, t3, t4, t5]
          ↓
        model
```

So:

```text
prefill cost = prompt length
```

For A:

```text
cost(A) = 5
```

---

## Decode cost

After prefill, the previous tokens are already represented in the KV cache.

We only feed the newest generated token:

```text
KV cache + [latest token]
               ↓
             model
```

Therefore:

```text
decode cost = 1
```

After A has completed prefill:

```text
cost(A) changes:

5 → 1
```

This is an important scheduler idea:

> The cost of a request can change while the request is alive.

---

# 4. One inference step

Our actual model execution remains separate from scheduling.

```python
@torch.no_grad()
def run_request(request):

    # First execution = prefill
    if not request["prefilled"]:

        outputs = model(
            input_ids=request["input_ids"],
            past_key_values=request["cache"],
            use_cache=True
        )

        request["prefilled"] = True

    # Every later execution = decode
    else:

        last_token = torch.tensor(
            [[request["generated_tokens"][-1]]],
            device=device
        )

        outputs = model(
            input_ids=last_token,
            past_key_values=request["cache"],
            use_cache=True
        )

    # Save the updated KV cache
    request["cache"] = outputs.past_key_values

    # Only the newest position predicts the next token
    next_token_logits = outputs.logits[:, -1, :]

    # Greedy decoding for now
    next_token = torch.argmax(
        next_token_logits,
        dim=-1
    ).item()

    request["generated_tokens"].append(next_token)

    # Stop when EOS appears or the generation
    # reaches its requested token limit
    if (
        next_token == tokenizer.eos_token_id
        or
        len(request["generated_tokens"])
        >= request["max_new_tokens"]
    ):
        request["finished"] = True
```

The separation remains:

```text
scheduler
    ↓
decides WHO gets work

run_request()
    ↓
performs ONE unit of model work
```

---

# 5. `running` and `scheduled_this_step` are different

This is the biggest conceptual change from our first scheduler.

Previously:

```text
running
```

basically meant:

> This request will execute.

Now we separate:

```text
running
```

from:

```text
scheduled_this_step
```

### `running`

Requests currently active in the engine.

### `scheduled_this_step`

Requests that actually receive model execution during **this particular scheduler iteration**.

For example:

```text
running = [A, B, C]

scheduled_this_step = [A, C]
```

B still exists.

B is still active.

B has not finished.

It simply receives no inference work during this scheduler step.

This distinction appears constantly in real scheduling systems:

```text
active ≠ executing right now
```

---

# 6. Resource-aware `schedule()`

We start each scheduler iteration with a fresh budget:

```python
def schedule():

    scheduled_this_step = []

    token_budget = max_tokens_per_step
```

With our configuration:

```text
token_budget = 16
```

The scheduler will consume this budget while selecting work.

---

# 7. First consider already-running requests

```python
    for request in running:

        cost = tokens_needed(request)

        if cost <= token_budget:

            scheduled_this_step.append(request)

            token_budget -= cost
```

Suppose:

```text
running = [A, B]

A decoding → cost 1
B decoding → cost 1

budget = 16
```

Then:

```text
A selected
budget = 15

B selected
budget = 14
```

Both continue decoding.

---

# 8. Admit new requests

After active requests are considered, we can spend the remaining budget on waiting requests.

```python
    index = 0

    while (
        index < len(waiting)
        and len(running) < max_running
    ):

        request = waiting[index]

        cost = tokens_needed(request)

        if cost <= token_budget:

            waiting.pop(index)

            running.append(request)

            scheduled_this_step.append(request)

            token_budget -= cost

        else:

            index += 1
```

The interesting part is:

```python
else:
    index += 1
```

We do **not** immediately stop when one request does not fit.

Instead:

> Check whether another waiting request can make useful progress.

---

# Strict FIFO vs our version

Suppose:

```text
waiting:

A cost 6
B cost 12
C cost 3

budget = 10
```

Strict FIFO:

```text
A fits
budget = 4

B doesn't fit

STOP

C doesn't get considered
```

Our work-conserving version:

```text
A fits
budget = 4

B doesn't fit
skip for this step

C costs 3
C fits

budget = 1
```

So:

```text
scheduled_this_step = [A, C]
```

and:

```text
B remains waiting
```

This improves utilization, although—as we will discuss later—it introduces its own fairness problem.

---

# 9. Oversized requests and forward progress

Now consider:

```text
max_tokens_per_step = 16
```

but:

```text
D prompt length = 40
```

Our cost function says:

```text
tokens_needed(D) = 40
```

So every scheduler iteration would ask:

```text
40 <= 16 ?
```

and get:

```text
False
```

D could become permanently stuck.

The proper solution is **chunked prefill**:

```text
40-token prompt

step 1 → process chunk
step 2 → process next chunk
step 3 → finish remaining prefill
```

But chunked prefill belongs later.

So for now we add a temporary forward-progress rule.

```python
    if (
        len(scheduled_this_step) == 0
        and len(waiting) > 0
        and len(running) < max_running
    ):

        request = waiting.pop(0)

        running.append(request)

        scheduled_this_step.append(request)
```

Meaning:

> If the scheduler cannot find anything at all to execute, allow the oldest request through even if it exceeds our toy token budget.

This avoids deadlock.

It is important to understand that this is a **temporary educational fallback**, not our final resource-management design.

---

# 10. Complete scheduler

Putting those pieces together:

```python
def schedule():

    scheduled_this_step = []

    token_budget = max_tokens_per_step


    # -----------------------------------
    # 1. Consider already-running requests
    # -----------------------------------

    for request in running:

        cost = tokens_needed(request)

        if cost <= token_budget:

            scheduled_this_step.append(request)

            token_budget -= cost


    # -----------------------------------
    # 2. Admit new requests if there is room
    # -----------------------------------

    index = 0

    while (
        index < len(waiting)
        and len(running) < max_running
    ):

        request = waiting[index]

        cost = tokens_needed(request)

        if cost <= token_budget:

            waiting.pop(index)

            running.append(request)

            scheduled_this_step.append(request)

            token_budget -= cost

        else:

            # Keep searching instead of letting one
            # expensive request block everyone behind it
            index += 1


    # -----------------------------------
    # 3. Guarantee forward progress
    # -----------------------------------

    if (
        len(scheduled_this_step) == 0
        and len(waiting) > 0
        and len(running) < max_running
    ):

        request = waiting.pop(0)

        running.append(request)

        scheduled_this_step.append(request)


    return scheduled_this_step
```

---

# 11. Engine loop

The engine repeatedly asks the scheduler what should execute:

```python
def run_scheduler():

    while len(waiting) > 0 or len(running) > 0:

        # Decide which requests get model work now
        scheduled_this_step = schedule()

        # Execute one inference step for each selected request
        for request in scheduled_this_step:

            run_request(request)

        # Remove requests that completed
        for request in running[:]:

            if request["finished"]:

                running.remove(request)

                finished.append(request)
```

Then:

```python
run_scheduler()
```

The overall control flow is now:

```text
                    ┌──────────────────────┐
                    │      WAITING         │
                    │   A    B    C ...    │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │      SCHEDULER       │
                    │                      │
                    │ arrival preference   │
                    │ request capacity     │
                    │ token budget         │
                    │ current step cost    │
                    └──────────┬───────────┘
                               │
                               ▼
                    scheduled_this_step
                               │
                               ▼
                    ┌──────────────────────┐
                    │   MODEL EXECUTION    │
                    │   one step/request   │
                    └──────────┬───────────┘
                               │
                               ▼
                         update state
                               │
                   ┌───────────┴───────────┐
                   │                       │
             still active              finished
                   │                       │
                   ▼                       ▼
               RUNNING                 FINISHED
                   │
                   └──────────────→ scheduler again
```

---

# Actual dry run of our example

Now let's trace the exact requests we just executed.

We have:

```text
max_running = 2

max_tokens_per_step = 16
```

Requests:

```text
A
prompt cost = 5
max_new_tokens = 10

B
prompt cost = 8
max_new_tokens = 10

C
prompt cost = 5
max_new_tokens = 10
```

Initially:

```text
WAITING
[A(5), B(8), C(5)]

RUNNING
[]

FINISHED
[]
```

The number in parentheses is the next-step token cost.

---

## Scheduler iteration 1

Start:

```text
token_budget = 16
```

There are no running requests yet.

So we start admitting from `waiting`.

### Consider A

```text
A cost = 5

5 <= 16
```

A fits.

Move:

```text
A:

waiting → running
```

Budget becomes:

```text
16 - 5 = 11
```

State:

```text
WAITING
[B(8), C(5)]

RUNNING
[A(5)]

SCHEDULED THIS STEP
[A]

budget = 11
```

---

### Consider B

B costs:

```text
8
```

and:

```text
8 <= 11
```

so B also fits.

Move:

```text
B:

waiting → running
```

Budget:

```text
11 - 8 = 3
```

Now:

```text
WAITING
[C(5)]

RUNNING
[A(5), B(8)]

SCHEDULED THIS STEP
[A, B]

budget = 3
```

We stop admitting because:

```text
len(running) = 2
max_running = 2
```

There is no active-request slot left.

---

## Execute iteration 1

A performs prefill:

```text
5 prompt tokens
      ↓
     model
      ↓
KV cache length = 5
      ↓
predict generated token #1
```

B performs prefill:

```text
8 prompt tokens
      ↓
     model
      ↓
KV cache length = 8
      ↓
predict generated token #1
```

After that:

```text
A generated = 1 token
B generated = 1 token
```

Most importantly, both requests are now:

```python
prefilled = True
```

Therefore their next costs change.

Before:

```text
A cost = 5
B cost = 8
```

Now:

```text
A cost = 1
B cost = 1
```

because they have entered decode.

---

# Scheduler iteration 2

Current state:

```text
WAITING
[C(5)]

RUNNING
[A(1), B(1)]

FINISHED
[]
```

Fresh budget:

```text
16
```

First consider existing running requests.

### A

```text
cost = 1
```

Schedule A:

```text
budget = 15
```

### B

```text
cost = 1
```

Schedule B:

```text
budget = 14
```

So:

```text
scheduled_this_step = [A, B]
```

Could we admit C?

No.

Not because of token budget.

We have plenty:

```text
budget = 14
C only needs = 5
```

The problem is:

```text
len(running) = 2
max_running = 2
```

There is no active-request slot available.

So C remains waiting.

---

## Execution iteration 2

A:

```text
feed generated token #1
↓
cache: 5 → 6
↓
predict token #2
```

B:

```text
feed generated token #1
↓
cache: 8 → 9
↓
predict token #2
```

State becomes:

```text
A generated = 2
B generated = 2
C still waiting
```

---

# Iterations 3 through 9

The same pattern repeats.

```text
RUNNING
[A, B]

WAITING
[C]
```

Every scheduler iteration:

```text
A costs 1
B costs 1

total cost = 2

token budget = 16
```

so both easily fit.

Conceptually:

```text
Iteration 3
A → token 3
B → token 3

Iteration 4
A → token 4
B → token 4

...

Iteration 9
A → token 9
B → token 9
```

C still cannot enter because:

```text
running slots = 2 / 2
```

---

# Scheduler iteration 10

Again:

```text
scheduled_this_step = [A, B]
```

Each gets one decode step.

A predicts token #10.

B predicts token #10.

Now:

```text
len(A.generated_tokens) = 10
len(B.generated_tokens) = 10
```

Their stopping condition becomes true:

```python
len(request["generated_tokens"]) >= request["max_new_tokens"]
```

so:

```text
A.finished = True
B.finished = True
```

The engine then removes both from `running`.

State:

```text
WAITING
[C]

RUNNING
[]

FINISHED
[A, B]
```

Two active slots have now become free.

---

# Scheduler iteration 11

Fresh:

```text
token_budget = 16
```

`running` is empty.

The scheduler sees C:

```text
C prefilled = False

cost(C) = 5
```

Check:

```text
5 <= 16
```

so C is admitted.

State:

```text
WAITING
[]

RUNNING
[C]

SCHEDULED THIS STEP
[C]

FINISHED
[A, B]

remaining budget = 11
```

C now performs prefill:

```text
5 prompt tokens
      ↓
     model
      ↓
cache = 5
      ↓
predict token #1
```

After that:

```text
cost(C) changes:

5 → 1
```

---

# Scheduler iterations 12 through 20

Now C is decoding.

Every iteration:

```text
cost(C) = 1
```

and it gets another token:

```text
Iteration 12 → token 2
Iteration 13 → token 3
Iteration 14 → token 4
...
Iteration 20 → token 10
```

At iteration 20:

```text
generated_tokens = 10
```

so:

```text
C.finished = True
```

and C moves:

```text
running → finished
```

Final state:

```text
WAITING
[]

RUNNING
[]

FINISHED
[A, B, C]
```

The outer loop checks:

```python
while len(waiting) > 0 or len(running) > 0
```

which becomes:

```text
False OR False
```

so the scheduler exits.

---

# Full request-flow diagram for our run

```text
START

waiting:
[A(5), B(8), C(5)]

running:
[]

                 token budget = 16
                         │
                         ▼
                 ┌─────────────┐
                 │ Scheduler 1 │
                 └──────┬──────┘
                        │
            A costs 5 ──┤ admit
                        │
            B costs 8 ──┤ admit
                        │
            C blocked by max_running
                        │
                        ▼

waiting:  [C]
running:  [A, B]

                        │
                        ▼
                 ┌─────────────┐
                 │   PREFILL   │
                 │   A + B     │
                 └──────┬──────┘
                        │
                        ▼

A: generated 1
B: generated 1

A cost becomes 1
B cost becomes 1

                        │
                        ▼

              Scheduler iterations
                    2 ... 10

                 ┌───────────┐
                 │ A decode  │
                 │ B decode  │
                 └─────┬─────┘
                       │
                       ▼

                 A reaches 10
                 B reaches 10
                       │
                       ▼

finished: [A, B]
running:  []
waiting:  [C]

                       │
                       ▼
                ┌──────────────┐
                │ Scheduler 11 │
                └──────┬───────┘
                       │
                    admit C
                       │
                       ▼

running: [C]

                       │
                       ▼
                   C prefill
                       │
                       ▼
                  C cost = 1
                       │
                       ▼

              Scheduler iterations
                   12 ... 20

                       │
                     decode
                       │
                       ▼

               C reaches 10
                       │
                       ▼

waiting:  []
running:  []
finished: [A, B, C]

                       │
                       ▼

                     STOP
```

---

# Final output

We printed:

```python
for request in finished:

    text = tokenizer.decode(
        request["generated_tokens"],
        skip_special_tokens=True
    )

    print(f"Request {request['id']}")

    print(f"Output: {text}")

    print(
        f"Generated tokens: "
        f"{len(request['generated_tokens'])}"
    )

    print(
        f"Cache length: "
        f"{request['cache'].get_seq_length()}"
    )

    print("-" * 50)
```

Our actual result was:

```text
Request A
Output: Paris. It is the largest city in Europe and
Generated tokens: 10
Cache length: 14
--------------------------------------------------

Request B
Output: ____ A. Venus B. Mars
Generated tokens: 10
Cache length: 17
--------------------------------------------------

Request C
Output: the cheetah, which can run at a
Generated tokens: 10
Cache length: 14
--------------------------------------------------
```

---

# Why the cache lengths are 14, 17 and 14

A had roughly:

```text
5 prompt tokens
```

and generated:

```text
10 tokens
```

But the final generated token has only been **predicted**.

It has not yet been sent back through the model.

Therefore:

```text
cache length
=
prompt tokens
+
generated tokens already processed

=
prompt length
+
10 - 1
```

For A:

```text
5 + 9 = 14
```

For B:

```text
8 + 9 = 17
```

For C:

```text
5 + 9 = 14
```

So the cache values are consistent with our inference loop.

---

# Important model-output disclaimer

This section is testing **scheduler behavior**, not model quality.

Request B produced:

```text
____ A. Venus B. Mars
```

for:

```text
The largest planet in our solar system is
```

which obviously does not make semantic sense.

The correct planet is Jupiter.

The issue is not caused by our scheduler.

We are using:

```text
Qwen/Qwen2.5-0.5B-Instruct
```

while directly tokenizing plain text:

```python
tokenizer(prompt)
```

rather than consistently using the formatting expected by the instruction-tuned model.

Because of that, the model can treat the prompt as arbitrary text continuation rather than as a clean user instruction.

That can produce strange continuations such as:

```text
____ A. Venus B. Mars...
```

From the next part onward, we should verify before building:

```text
model type
      +
expected input format
      +
chat template / tokenizer compatibility
```

This prevents a bad prompting setup from being confused with an inference-engine bug.

---

# Limitations of our current scheduler

This scheduler is much closer to the right **control-flow mental model**, but it is still intentionally incomplete.

## Limitation 1 — Selected requests are not actually batched

We have:

```python
for request in scheduled_this_step:
    run_request(request)
```

So if:

```text
scheduled_this_step = [A, B]
```

execution is:

```text
model(A)
then
model(B)
```

These are still two separate forward passes.

The GPU is not executing:

```text
[A, B] → one model forward
```

yet.

### How this gets solved

**Part 5: Continuous Batching**

There we will start turning:

```text
scheduled requests
```

into actual changing batches that execute together.

---

# Limitation 2 — Token budget is only a simplified resource model

We use:

```python
max_tokens_per_step = 16
```

but production resource consumption depends on much more than simply:

```text
number of tokens
```

There are considerations such as:

```text
KV-cache memory
model memory
number of active sequences
prefill workload
decode workload
GPU kernel efficiency
cache blocks
```

So:

```text
token budget
```

is a useful scheduling abstraction, not a complete GPU-memory model.

### How this gets solved

Later parts will introduce proper **KV-cache management** and eventually a **paged/block allocator**.

Then the scheduler can make decisions based on actual cache capacity rather than only a toy integer budget.

---

# Limitation 3 — Large prefills can exceed our entire budget

Suppose:

```text
max_tokens_per_step = 16
```

and:

```text
prompt length = 500
```

Our current request asks for:

```text
cost = 500
```

all at once.

That does not fit.

Our temporary fallback simply allows one oversized request through when nothing else can execute.

That technically violates:

```text
max_tokens_per_step
```

so the budget is not a hard guarantee.

### How this gets solved

Later we will implement **chunked prefill**.

Instead of:

```text
500-token prefill
→ process all 500 now
```

we can do something like:

```text
step 1 → some prompt tokens
step 2 → next prompt chunk
step 3 → next chunk
...
```

so large prefills can coexist with other work without breaking the scheduler budget.

---

# Limitation 4 — Skipping expensive requests can cause starvation

Our work-conserving scheduler can do:

```text
B doesn't fit
→ skip B

C fits
→ run C
```

That improves utilization.

But imagine new tiny requests keep arriving behind B:

```text
B = expensive

C = cheap
D = cheap
E = cheap
F = cheap
...
```

If we always prefer requests that fit, B could potentially wait for a very long time.

This is called **starvation**.

So there is a tradeoff:

```text
strict FIFO
    ↓
strong arrival-order fairness
but possibly worse utilization

work-conserving skipping
    ↓
better utilization
but potentially worse fairness
```

### How production systems deal with this

Schedulers can add mechanisms such as:

```text
aging
priority adjustments
fairness rules
maximum waiting time
reservations
different queue policies
```

We do not need those yet.

Our goal here is simply to see that scheduling always involves tradeoffs between:

```text
latency
throughput
fairness
utilization
```

---

# Limitation 5 — We do not have real priority scheduling

Our scheduler is:

```text
arrival-order biased
+
resource-aware
```

It does **not** have:

```text
request.priority
```

or any explicit service classes.

A truly priority-aware scheduler might receive:

```text
A priority = 5
B priority = 1
C priority = 3
```

and consider B before A regardless of arrival order.

### How this could be added

Later, request metadata could contain:

```python
"priority": priority
```

and scheduler ordering could use it.

But implementing every scheduling policy is not the goal of this inference-engine project.

The important point is:

```text
policy
```

and:

```text
resource constraint
```

are separate concepts.

---

# Limitation 6 — No real preemption

Suppose a request is currently active and owns lots of KV-cache memory.

A production scheduler may sometimes need to:

```text
pause it
release/reclaim resources
run another request
resume the first request later
```

Our current scheduler does not do this.

Once a request enters `running`, its `DynamicCache` simply stays attached to that Python object until it finishes.

### Why we are not faking this now

Real preemption only becomes interesting once we have actual resource ownership such as:

```text
KV-cache blocks
```

If we merely wrote:

```python
running.remove(request)
waiting.append(request)
```

and called that preemption, we would mostly be moving Python objects around.

We would not actually be reclaiming meaningful GPU resources.

### How this gets solved

After we build real KV-cache management, preemption will become much more meaningful.

---

# Limitation 7 — Each request owns a `DynamicCache`

Right now:

```python
request["cache"] = DynamicCache()
```

gives every request its own independent cache object.

That is perfect for learning.

But production inference engines usually need much tighter control over KV-cache memory.

They cannot simply let every request independently grow arbitrary cache tensors forever.

### How this gets solved

Later:

```text
KV-cache management
       ↓
block allocation
       ↓
paged KV cache
```

will replace this much simpler ownership model.

---

# Limitation 8 — Requests do not arrive dynamically yet

All three requests are inserted before:

```python
run_scheduler()
```

starts.

Real servers look more like:

```text
A arrives
scheduler already running

B arrives later
scheduler already running

C arrives even later
```

The waiting queue changes while generation is happening.

Our scheduler control flow already prepares us for this idea, but the notebook still uses a fixed request list for clarity.

### How this gets solved

Continuous batching naturally makes this much more important because new requests can join while existing requests are decoding.

---

# What we learned from Part 4

We started with:

```text
generate one request
```

and ended with a real scheduling control loop:

```text
requests arrive
      ↓
waiting queue
      ↓
scheduler examines state
      ↓
consider request order
      ↓
consider active sequence capacity
      ↓
consider token-work budget
      ↓
select scheduled_this_step
      ↓
execute one model step
      ↓
update per-request state
      ↓
finished requests leave
      ↓
new requests become eligible
      ↓
repeat
```

The scheduler now understands that:

```text
request exists
```

does not mean:

```text
request is active
```

and:

```text
request is active
```

does not necessarily mean:

```text
request receives GPU work this iteration
```

Those are three different states:

```text
WAITING
    request exists but is not active

RUNNING
    request is active

SCHEDULED_THIS_STEP
    request actually receives model work now
```

That is the most important mental model from this part.

---

# Final architecture after Part 4

```text
                       incoming requests
                              │
                              ▼
                    ┌──────────────────┐
                    │     WAITING      │
                    └────────┬─────────┘
                             │
                             ▼
              ┌────────────────────────────┐
              │         SCHEDULER          │
              │                            │
              │ • arrival-order preference │
              │ • max active requests      │
              │ • token-work budget        │
              │ • prefill/decode cost      │
              │ • work-conserving choice   │
              └─────────────┬──────────────┘
                            │
                            ▼
                 scheduled_this_step
                            │
                            ▼
                  ┌──────────────────┐
                  │ MODEL EXECUTION  │
                  │                  │
                  │ currently still  │
                  │ one request at   │
                  │ a time           │
                  └────────┬─────────┘
                           │
                           ▼
                  update request state
                           │
              ┌────────────┴────────────┐
              │                         │
              ▼                         ▼
         still active               finished
              │                         │
              ▼                         ▼
          RUNNING                   FINISHED
              │
              └──────────→ SCHEDULER
```

The glaring remaining weakness is now:

```text
scheduled_this_step = [A, B]

but execution is still:

model(A)
model(B)
```

rather than:

```text
[A, B]
   ↓
one batched model execution
```

That is exactly where **Part 5 - Continuous Batching** begins.
